# Chapter 15 - Monitoring Unpaid Claim Estimates

> Deviations of actual development from projected development of claims or
> claim counts are one of the most useful diagnostic tools for evaluating the
> accuracy of the unpaid claim estimate.
>
> -- Friedland, Chapter 15

The last part of Chapter 15 is a **roll-forward**: take the ultimates and
reporting pattern selected at one valuation, and compare actual reported
claims in the next period with the amount that pattern said should emerge.

This notebook recreates Friedland's **DC Insurer** monitoring exhibits
(*Exhibit IV, Sheets 1-4*). `friedland_dc_insurer` is the quarterly reported
triangle through 36 months, plus the 12/31/2007 and 12/31/2008 diagonals.
Selected CDFs stop at 36 months (age-to-ult 1.000), so later ages are treated
as fully reported.

Expected emergence in the next calendar period comes from the fitted
`Chainladder` model: `full_triangle_.dev_to_val()` at the later valuation
minus the prior `latest_diagonal`. That is the same pattern as the gallery
Actual vs Expected example. For each accident year that is the Friedland
formula

$$
\frac{\text{Ultimate}_{t_0} - \text{Reported}_{t_0}}{1 - p_{t_0}}
\times (p_{t_1} - p_{t_0})
$$

where $p_t = 1 / \text{CDF}_t$.

In [ ]:
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


def as_series(tri):
    s = tri.to_frame(origin_as_datetime=False).iloc[:, 0]
    s.index = [int(getattr(i, "year", i)) for i in s.index]
    return s

## Exhibit IV, Sheet 1 - Reported claims triangle

DC Insurer reports quarterly. The sample carries ages 3 through 36, which is
the printed Sheet 1 triangle. The selected tail is 1.000 at 36 months, so
older accident years are carried at that age-36 value through 12/31/2007.

In [ ]:
tri = cl.load_sample("friedland_dc_insurer")
tri = tri[tri.origin < "2008"]
tri_2007 = tri[tri.valuation < "2008"]

selected_cdf = {
    3: 4.125,
    6: 2.177,
    9: 1.487,
    12: 1.136,
    15: 1.016,
    18: 1.008,
    21: 1.003,
    24: 1.001,
    27: 1.000,
    30: 1.000,
    33: 1.000,
    36: 1.000,
}
for age in (int(a) for a in tri_2007.ddims):
    selected_cdf.setdefault(age, 1.000)

dev = cl.DevelopmentConstant(patterns=selected_cdf, style="cdf").fit_transform(tri_2007)
model = cl.Chainladder().fit(dev)

display(tri_2007[tri_2007.development <= 36])
display(dev.cdf_[dev.cdf_.ddims <= 36])

## Exhibit IV, Sheet 2 - Ultimates at 12/31/2007

DC Insurer selects ultimates with the reported development technique. Age 12
uses a 1.136 CDF (88.0% reported); age 24 uses 1.001 (99.9% reported); age 36
and later are 1.000.

`model_diagnostics` pulls Latest, CDF, and Ultimate from the fitted
`Chainladder` model. The text's worked example for accident year 2007 is
$2{,}463 \times 1.136 = 2{,}798$.

In [ ]:
diag = cl.model_diagnostics(model)
years = [int(year) for year in tri_2007.origin.year]
sheet2 = pd.DataFrame(index=years)
sheet2["Age"] = (2007 - sheet2.index + 1) * 12
sheet2["Reported at 12/31/07"] = as_series(diag["Latest"])
sheet2["CDF to Ultimate"] = np.round(as_series(diag["CDF"]), 3)
sheet2["Projected Ultimate"] = np.round(as_series(diag["Ultimate"]), 0)
display(sheet2)
print(f"Total projected ultimate: {sheet2['Projected Ultimate'].sum():,.0f}")

## Exhibit IV, Sheet 3 - Annual monitoring test

One year later, compare calendar-year 2008 actual reported claims with the
amount implied by the 12/31/2007 model. Slice `full_triangle_` at the 2008
valuation for the expected cumulative; subtract the 2007 `latest_diagonal`
for expected emergence.

The text works accident year 2007 as

$$
\frac{2{,}798 - 2{,}463}{1 - 0.880} \times (0.999 - 0.880) = 332
$$

and accident year 2006 as

$$
\frac{2{,}952 - 2{,}949}{1 - 0.999} \times (1.000 - 0.999) = 3.
$$

Older years are fully reported, so expected emergence is zero.

In [ ]:
expected_cum = model.full_triangle_.dev_to_val()
expected_cum = expected_cum[expected_cum.valuation == tri.valuation_date]

reported_2007 = as_series(tri_2007.latest_diagonal)
reported_2008 = as_series(tri.latest_diagonal)
expected = np.round(as_series(expected_cum) - reported_2007, 0)
actual = reported_2008 - reported_2007

pct_2007 = 1.0 / as_series(diag["CDF"]).to_numpy()
cdf_2008 = np.array([selected_cdf.get(int(age) + 12, 1.000) for age in sheet2["Age"]])
pct_2008 = 1.0 / cdf_2008

sheet3 = pd.DataFrame(index=years)
sheet3["Selected Ultimate"] = sheet2["Projected Ultimate"]
sheet3["% Reported 12/31/07"] = np.round(pct_2007, 3)
sheet3["% Reported 12/31/08"] = np.round(pct_2008, 3)
sheet3["Reported 12/31/07"] = reported_2007
sheet3["Reported 12/31/08"] = reported_2008
sheet3["Actual"] = actual
sheet3["Expected"] = expected
sheet3["Difference"] = actual - expected
display(sheet3)
display(sheet3[["Actual", "Expected", "Difference"]].sum().rename("Total").to_frame().T)

## Exhibit IV, Sheet 4 - Monthly monitoring test

DC Insurer has quarterly development factors. Monthly percent-reported values
are **linear interpolations of the quarterly percent reported**. Between age 12
(88.0%) and age 15 ($1 / 1.016 \approx 98.4%$) that gives 91.5% at 13 months
and 95.0% at 14 months, matching the printed January / February 2008 template.

Expected monthly emergence uses the fitted 12/31/2007 `ibnr_` and those
interpolated percents for **both** January and February:
$\text{IBNR}_{12/31/07} \times (p_{t+1} - p_t) / (1 - p_{12/31/07})$.
That is Friedland's column notes (10) and (13): February does not rebase on
January actuals. The quarterly triangle cannot hold monthly CDFs, so the
interpolation stays on the selected pattern rather than on `DevelopmentConstant`.

In [ ]:
def pct_reported_at(age):
    """Linearly interpolate percent reported between quarterly CDF ages."""
    knots = np.array(sorted(selected_cdf))
    pcts = 1.0 / np.array([selected_cdf[k] for k in knots])
    if age <= knots[0]:
        return float(pcts[0])
    if age >= knots[-1]:
        return 1.0
    return float(np.interp(age, knots, pcts))


pct_jan = np.array([pct_reported_at(age + 1) for age in sheet2["Age"]])
pct_feb = np.array([pct_reported_at(age + 2) for age in sheet2["Age"]])

# Printed latest reported at 1/31/08 and 2/29/08 for the two immature years.
reported_jan = reported_2007.copy()
reported_feb = reported_2007.copy()
reported_jan.loc[2007] = 2473
reported_feb.loc[2007] = 2538
reported_jan.loc[2006] = 2951
reported_feb.loc[2006] = 2986
reported_jan.loc[2005] = 2825
reported_feb.loc[2005] = 2832
reported_jan.loc[2004] = 3422
reported_feb.loc[2004] = 3422
reported_feb.loc[2003] = 2998
reported_jan.loc[2001] = 2096
reported_feb.loc[2001] = 2096

ibnr = as_series(model.ibnr_)
unreported = 1.0 - pct_2007
scale = np.divide(
    ibnr.to_numpy(), unreported, out=np.zeros(len(ibnr)), where=unreported > 0
)
expected_jan = np.round(scale * (pct_jan - pct_2007), 0)
expected_feb = np.round(scale * (pct_feb - pct_jan), 0)

actual_jan = reported_jan - reported_2007
actual_feb = reported_feb - reported_jan

sheet4 = pd.DataFrame(index=years)
sheet4["Selected Ultimate"] = sheet2["Projected Ultimate"]
sheet4["% Reported 12/31/07"] = np.round(pct_2007, 3)
sheet4["% Reported 1/31/08"] = np.round(pct_jan, 3)
sheet4["% Reported 2/29/08"] = np.round(pct_feb, 3)
sheet4["Reported 12/31/07"] = reported_2007
sheet4["Reported 1/31/08"] = reported_jan
sheet4["Reported 2/29/08"] = reported_feb
sheet4["Actual Jan"] = actual_jan
sheet4["Expected Jan"] = expected_jan
sheet4["Diff Jan"] = actual_jan - expected_jan
sheet4["Actual Feb"] = actual_feb
sheet4["Expected Feb"] = expected_feb
sheet4["Diff Feb"] = actual_feb - expected_feb
display(sheet4)
display(
    sheet4[
        [
            "Actual Jan",
            "Expected Jan",
            "Diff Jan",
            "Actual Feb",
            "Expected Feb",
            "Diff Feb",
        ]
    ]
    .sum()
    .rename("Total")
    .to_frame()
    .T
)

## Reconciliation

In [ ]:
# Exhibit IV, Sheet 1
sheet1 = tri_2007[tri_2007.development <= 36].to_frame(origin_as_datetime=False)
sheet1.index = [int(getattr(i, "year", i)) for i in sheet1.index]
assert sheet1.loc[1997, 3] == 861
assert sheet1.loc[2007, 12] == 2463
assert sheet1.loc[2006, 24] == 2949

# Exhibit IV, Sheet 2
assert sheet2.loc[2007, "Projected Ultimate"] == 2798
assert sheet2.loc[2006, "Projected Ultimate"] == 2952

# Exhibit IV, Sheet 3
assert sheet3.loc[2007, "% Reported 12/31/07"] == 0.880
assert sheet3.loc[2007, "% Reported 12/31/08"] == 0.999
assert sheet3.loc[2007, "Expected"] == 332
assert sheet3.loc[2006, "Expected"] == 3
assert sheet3["Expected"].sum() == 335

# Exhibit IV, Sheet 4 - interpolated percent reported and expected emergence
assert np.isclose(sheet4.loc[2007, "% Reported 1/31/08"], 0.915, atol=5e-4)
assert np.isclose(sheet4.loc[2007, "% Reported 2/29/08"], 0.950, atol=5e-4)
assert sheet4.loc[2007, "Expected Jan"] == 97
assert sheet4.loc[2007, "Expected Feb"] == 97
assert sheet4.loc[2006, "Expected Jan"] == 1
assert sheet4.loc[2006, "Expected Feb"] == 1